In [ ]:
import gradio as gr
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


In [ ]:

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [ ]:
system_message = """
You are a data generator that creates synthetic but realistic Spotify usage data.

Rules:
- Output ONLY valid JSON
- Do NOT include explanations or extra text
- Ensure all numeric values are realistic
"""

In [ ]:
def build_prompt(num_rows, fields):
    field_list = "\n".join([f"- {f}" for f in fields])

    prompt = f"""
    Generate {num_rows} Spotify users in JSON format.

    Each object must contain:
    - user_id
    {field_list}

    Ensure values are realistic.

    Return only JSON.
    """
    return prompt

In [ ]:

def generate_data(num_rows, fields):
    response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": build_prompt(num_rows, fields)}
    ],
    response_format={"type": "json_object"}
    )
    data = response.choices[0].message.content
    data = json.loads(data)
    print(data)
    # Handle different GPT outputs
    
    if "users" in data:
        users = data["users"]
    elif isinstance(data, list):
        users = data
    elif isinstance(data, dict):
        users = [data]   
    else:
        raise ValueError("Unexpected GPT output")

    df = pd.DataFrame(users)  
    return df  



In [ ]:
def generate_and_download(num_rows, fields):
    df = generate_data(num_rows, fields)
    file_path = "spotify_data.csv"
    df.to_csv(file_path, index=False)
    return df, file_path

In [ ]:

with gr.Blocks() as ui:
    gr.Markdown("## Spotify Synthetic Data Generator")

    with gr.Row():
        num_rows = gr.Number(label="Number of users", value=10)

        fields = gr.CheckboxGroup(
            choices=[
                "user_name",
                "age",
                "location",
                "subscription_type",
                "favorite_genre",
                "song_minutes_per_day",
                "podcast_minutes_per_day",
                "audiobook_minutes_per_day",
                "shares_songs"
            ],
            label="Select fields"
        )

    generate_btn = gr.Button("Generate Data")

    output_table = gr.Dataframe()
    download_file = gr.File(label="Download CSV")

    generate_btn.click(
        fn=generate_and_download,
        inputs=[num_rows, fields],
        outputs=[output_table, download_file]
    )

ui.launch()